# Data split approach

In [ ]:
import os
import sys
from pathlib import Path

import pandas as pd
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 80)

cwd = Path.cwd().resolve()
REPO_ROOT = cwd if (cwd / "pyproject.toml").exists() else cwd.parent
assert (REPO_ROOT / "pyproject.toml").exists(), "Could not locate the repository root."
sys.path.insert(0, str(REPO_ROOT))

# Edit this, or set BREXPERT_DATA_ROOT, if the data directory lives elsewhere.
DATA_ROOT = Path(os.environ.get("BREXPERT_DATA_ROOT", REPO_ROOT.parent / "data")).resolve()
print(f"DATA_ROOT = {DATA_ROOT}")

## Step 1 — Load aggregated processed data

In [ ]:
import glob

processed_csvs = sorted(glob.glob(str(DATA_ROOT / "processed" / "**" / "*.csv"), recursive=True))
agg_df = pd.concat([pd.read_csv(p) for p in processed_csvs], ignore_index=True)
print(f"{len(processed_csvs)} processed csvs, {len(agg_df)} rows, {agg_df['patient'].nunique()} unique patients")
agg_df[["id", "patient", "dataset", "modality", "birads"]].head()

## Step 2 — Collapse BI-RADS into action-oriented categories

In [ ]:
from scripts.split import convert_birads

agg_df["original_birads"] = agg_df["birads"]
agg_df["birads"] = agg_df["birads"].apply(convert_birads)
agg_df.groupby(["original_birads", "birads"]).size().rename("rows").reset_index()

## Step 3 — Patient-level label inventory (before splitting)

In [ ]:
patient_labels_df = agg_df[["patient", "modality", "birads"]].dropna().drop_duplicates()
label_counts = patient_labels_df.groupby(["modality", "birads"])["patient"].nunique().rename("patients")
print(f"{patient_labels_df['patient'].nunique()} unique patients across {len(label_counts)} (modality, birads) labels")
label_counts.sort_index()

## Step 4 — Split configuration

In [ ]:
import inspect

from utils.stratified_splitter import split_data_budgeted

split_defaults = inspect.signature(split_data_budgeted).parameters
print(f"val_total target: {split_defaults['val_total'].default} patients")
print(f"test_total target: {split_defaults['test_total'].default} patients")

## Step 5 — Run the greedy quota-based split

In [ ]:
import logging

logging.basicConfig(level=logging.INFO, format="%(message)s", force=True)

train_df, test_df, val_df = split_data_budgeted(agg_df)

## Step 6 — Resulting split sizes

In [ ]:
pd.DataFrame(
    {
        "patients": [df["patient"].nunique() for df in (train_df, val_df, test_df)],
        "rows": [len(df) for df in (train_df, val_df, test_df)],
    },
    index=["train", "val", "test"],
)

## Step 7 — Leakage and completeness sanity checks

In [ ]:
train_p, val_p, test_p = (set(df["patient"]) for df in (train_df, val_df, test_df))
print(f"train ∩ val: {len(train_p & val_p)} | train ∩ test: {len(train_p & test_p)} | val ∩ test: {len(val_p & test_p)}")

covered = train_p | val_p | test_p
print(f"patients covered by a split: {len(covered)} / {agg_df['patient'].nunique()} total")

## Step 8 — Per-modality, per-BI-RADS distribution across splits

In [ ]:
combined = pd.concat(
    [
        train_df.assign(split="train"),
        val_df.assign(split="val"),
        test_df.assign(split="test"),
    ]
)
combined.groupby(["modality", "birads", "split"])["id"].count().unstack(fill_value=0)